Before we start, we need to make sure that we have a Kafka cluster running and a topic that produces some streaming data. For simplicity, we will use a single-node Kafka cluster and a topic named `users`. Open the `4.0 user-gen-kafka.ipynb` notebook and execute the cell. This notebook produces a user record every few seconds and put it on a Kafka topic called users. 

In [4]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, avg
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [5]:
builder = (SparkSession.builder
           .appName("transform-filter-streaming")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "2g")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder,['org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1']).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [6]:
df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka:9092")
      .option("subscribe", "users")
      .option("startingOffsets", "earliest")
      .load())

In [7]:
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("country", StringType(), True),
])

df = df.withColumn("value", from_json(col("value").cast("STRING"), schema))

In [10]:
df = df.select(
    col('value.id').alias('id'),
    col('value.name').alias('name'),
    col('value.age').alias('age'),
    col('value.gender').alias('gender'),
    col('value.country').alias('country'))

In [11]:
df = (
    df.select(
        "age",
        "country",
        "gender"
    )
    .filter("age >= 21")
    .groupBy(
        "country",
        "gender"
    )
    .agg(
        avg("age")
        .alias("avg_age")
    )
    
)

In [12]:
query = (
     df.writeStream
    .outputMode("complete")
    .format("console")
    .start()
)

-------------------------------------------
Batch: 0
-------------------------------------------
+---------+------+------------------+
|  country|gender|           avg_age|
+---------+------+------------------+
|   Brazil|     F| 41.64705882352941|
|   Brazil|     M|             42.68|
|      USA|     F|             45.44|
|Australia|     F|              40.4|
|   Canada|     M| 46.17857142857143|
|       UK|     M| 40.88235294117647|
|      USA|     M| 43.05555555555556|
|    India|     M| 44.80952380952381|
|    China|     M| 42.57142857142857|
|    China|     F| 47.31428571428572|
|   Canada|     F|45.515151515151516|
|Australia|     M| 44.23809523809524|
|    India|     F|45.903225806451616|
|       UK|     F| 44.13636363636363|
+---------+------+------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+---------+------+------------------+
|  country|gender|           avg_age|
+---------+------+------------------+
|   Brazil|     F| 41.64705882352941|
|   Brazil|     M|             42.68|
|      USA|     F|             45.44|
|Australia|     F|              40.4|
|   Canada|     M| 46.17857142857143|
|       UK|     M| 40.88235294117647|
|      USA|     M| 43.10526315789474|
|    India|     M| 44.80952380952381|
|    China|     M| 42.57142857142857|
|    China|     F| 47.31428571428572|
|   Canada|     F|45.515151515151516|
|Australia|     M| 44.23809523809524|
|    India|     F|45.903225806451616|
|       UK|     F| 44.13636363636363|
+---------+------+------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+---------+------+------------------+
|  country|gender|           avg_age|
+---------+------+------------------+
|   Brazil|     F| 41.64705882352941|
|   Brazil|     M|             42.68|
|      USA|     F|             45.44|
|Australia|     F|              40.4|
|   Canada|     M| 46.17857142857143|
|       UK|     M| 40.88235294117647|
|      USA|     M| 43.10526315789474|
|    India|     M| 44.80952380952381|
|    China|     M| 42.57142857142857|
|    China|     F| 47.31428571428572|
|   Canada|     F|45.515151515151516|
|Australia|     M| 44.18181818181818|
|    India|     F|45.903225806451616|
|       UK|     F| 44.13636363636363|
+---------+------+------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+---------+------+------------------+
|  country|gender|           avg_age|
+---------+------+------------------+
|   Brazil|     F| 41.64705882352941|
|   Brazil|     M|             42.68|
|      USA|     F|             45.44|
|Australia|     F|              40.4|
|   Canada|     M| 46.17857142857143|
|       UK|     M| 40.88235294117647|
|      USA|     M| 43.10526315789474|
|    India|     M| 44.22727272727273|
|    China|     M| 42.57142857142857|
|    China|     F| 47.31428571428572|
|   Canada|     F|45.515151515151516|
|Australia|     M| 44.18181818181818|
|    India|     F|45.903225806451616|
|       UK|     F| 44.13636363636363|
+---------+------+------------------+



-------------------------------------------
Batch: 4
-------------------------------------------
+---------+------+------------------+
|  country|gender|           avg_age|
+---------+------+------------------+
|   Brazil|     F| 41.64705882352941|
|   Brazil|     M|             42.68|
|      USA|     F|             45.44|
|Australia|     F|              40.4|
|   Canada|     M| 46.17857142857143|
|       UK|     M| 40.88235294117647|
|      USA|     M| 43.10526315789474|
|    India|     M| 44.22727272727273|
|    China|     M| 42.57142857142857|
|    China|     F|46.638888888888886|
|   Canada|     F|45.515151515151516|
|Australia|     M| 44.18181818181818|
|    India|     F|45.903225806451616|
|       UK|     F| 44.13636363636363|
+---------+------+------------------+



...

-------------------------------------------
Batch: 12
-------------------------------------------
+---------+------+------------------+
|  country|gender|           avg_age|
+---------+------+------------------+
|   Brazil|     F| 41.77142857142857|
|   Brazil|     M|             42.68|
|      USA|     F|             45.44|
|Australia|     F|              40.4|
|   Canada|     M| 46.17857142857143|
|       UK|     M|              40.0|
|      USA|     M| 43.10526315789474|
|    India|     M| 44.22727272727273|
|    China|     M| 42.57142857142857|
|    China|     F|46.638888888888886|
|   Canada|     F|45.515151515151516|
|Australia|     M| 44.34782608695652|
|    India|     F|            45.125|
|       UK|     F|44.083333333333336|
+---------+------+------------------+



-------------------------------------------
Batch: 13
-------------------------------------------
+---------+------+------------------+
|  country|gender|           avg_age|
+---------+------+------------------+
|   Brazil|     F| 41.77142857142857|
|   Brazil|     M|             42.68|
|      USA|     F|             45.44|
|Australia|     F|              40.4|
|   Canada|     M| 46.17857142857143|
|       UK|     M|              40.0|
|      USA|     M| 43.10526315789474|
|    India|     M| 44.22727272727273|
|    China|     M| 42.57142857142857|
|    China|     F|46.638888888888886|
|   Canada|     F|45.515151515151516|
|Australia|     M| 44.34782608695652|
|    India|     F|            45.125|
|       UK|     F|44.083333333333336|
+---------+------+------------------+



-------------------------------------------
Batch: 14
-------------------------------------------
+---------+------+------------------+
|  country|gender|           avg_age|
+---------+------+------------------+
|   Brazil|     F| 41.77142857142857|
|   Brazil|     M| 42.38461538461539|
|      USA|     F|             45.44|
|Australia|     F|              40.4|
|   Canada|     M| 46.17857142857143|
|       UK|     M|              40.0|
|      USA|     M| 43.10526315789474|
|    India|     M| 44.22727272727273|
|    China|     M| 42.57142857142857|
|    China|     F|46.638888888888886|
|   Canada|     F|45.515151515151516|
|Australia|     M| 44.34782608695652|
|    India|     F|            45.125|
|       UK|     F|44.083333333333336|
+---------+------+------------------+



-------------------------------------------
Batch: 15
-------------------------------------------
+---------+------+------------------+
|  country|gender|           avg_age|
+---------+------+------------------+
|   Brazil|     F| 41.77142857142857|
|   Brazil|     M| 42.38461538461539|
|      USA|     F| 46.03846153846154|
|Australia|     F|              40.4|
|   Canada|     M| 46.17857142857143|
|       UK|     M|              40.0|
|      USA|     M| 43.10526315789474|
|    India|     M| 44.22727272727273|
|    China|     M| 42.57142857142857|
|    China|     F|46.638888888888886|
|   Canada|     F|45.515151515151516|
|Australia|     M| 44.34782608695652|
|    India|     F|            45.125|
|       UK|     F|44.083333333333336|
+---------+------+------------------+



25/02/27 12:09:40 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 16, writer: ConsoleWriter[numRows=20, truncate=true]] is aborting.
25/02/27 12:09:40 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 16, writer: ConsoleWriter[numRows=20, truncate=true]] aborted.
25/02/27 12:09:40 ERROR MicroBatchExecution: Query [id = 364d3ca6-fe2c-46ed-b932-b3c828e0e71b, runId = d3d7477c-406b-4279-b529-f99a4a412629] terminated with error
org.apache.spark.SparkException: Job 16 cancelled because SparkContext was shut down
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1(DAGScheduler.scala:1212)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$cleanUpAfterSchedulerStop$1$adapted(DAGScheduler.scala:1210)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:79)
	at org.apache.spark.scheduler.DAGScheduler.cleanUpAfterSchedulerStop(DAGScheduler.scala:1210)
	at org.apache.spark.scheduler.DAGSchedulerEvent